# DS2002 Capstone — EV Charging Station Analytics

**Team Number:** `team-14`

**Team Members:**
- Suhanee Singh (jza3au)
- Emily Friedman (hjp3zf)
- Tina Lin (suk3ck)
- Sydney Huang (nbw7ar)

**Date:** 2026-04-30

---

## Cloud Setup / GCS authentication

Authenticate to GCS, download raw data, and verify your team folder.

In [ ]:
# Install the GCS library (run once)
!pip install google-cloud-storage -q

In [ ]:
import os
import json
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import requests

from google.colab import auth
auth.authenticate_user()

from google.cloud import storage
client = storage.Client(project="ds2002-492012")
bucket = client.bucket("ds2002-capstone-sp26-v2")
print("GCS authentication successful.")

GCS authentication successful.


In [ ]:
# Download raw data from GCS
TEAM = "team-14"

files = [
    "raw-data/charging_sessions.csv",
    "raw-data/station_locations.csv",
    "raw-data/vehicle_types.csv",
    "raw-data/grid_operators.csv",
    "raw-data/energy_and_demand.db",
]

os.makedirs("data", exist_ok=True)
for f in files:
    blob = bucket.blob(f)
    local = os.path.join("data", os.path.basename(f))
    blob.download_to_filename(local)
    print(f"Downloaded {f}")

Downloaded raw-data/charging_sessions.csv
Downloaded raw-data/station_locations.csv
Downloaded raw-data/vehicle_types.csv
Downloaded raw-data/grid_operators.csv
Downloaded raw-data/energy_and_demand.db


## Data Loading

In [ ]:
sessions = pd.read_csv("data/charging_sessions.csv")
stations = pd.read_csv("data/station_locations.csv")
vehicles = pd.read_csv("data/vehicle_types.csv")
operators = pd.read_csv("data/grid_operators.csv")

conn = sqlite3.connect("data/energy_and_demand.db")
demand = pd.read_sql("SELECT * FROM daily_demand_summary", conn)
grid = pd.read_sql("SELECT * FROM grid_capacity_levels", conn)
conn.close()

print(f"Sessions: {sessions.shape}")
print(f"Stations: {stations.shape}")
print(f"Vehicles: {vehicles.shape}")
print(f"Operators: {operators.shape}")
print(f"Demand:   {demand.shape}")
print(f"Grid:     {grid.shape}")

Sessions: (27451, 11)
Stations: (21, 8)
Vehicles: (42, 6)
Operators: (5, 7)
Demand:   (6570, 7)
Grid:     (1825, 6)


## Data Exploration

Inspect each dataset before cleaning.

In [ ]:
# Inspect Raw Data:

sessions = pd.read_csv("data/charging_sessions.csv")
print("charging_sessions.csv")
print(f"  Shape: {sessions.shape}")
print(f"  Columns: {sessions.columns.tolist()}")
print(f"  Null counts:")
print(sessions.isnull().sum())
sessions.head()


charging_sessions.csv
  Shape: (27451, 11)
  Columns: ['session_id', 'station_id', 'vehicle_id', 'session_start', 'session_end', 'kwh_delivered', 'session_type', 'cost_usd', 'payment_method', 'connector_used', 'user_id']
  Null counts:
session_id         0
station_id         0
vehicle_id        77
session_start      0
session_end        0
kwh_delivered     90
session_type       0
cost_usd          67
payment_method     0
connector_used    66
user_id            0
dtype: int64


,session_id,station_id,vehicle_id,session_start,session_end,kwh_delivered,session_type,cost_usd,payment_method,connector_used,user_id
0,SES-005321,STN-013,VH-008,2025-03-16 17:06:25,2025-03-16 18:30:25,1.67,Level 1,0.53,credit card,J1772,U-4469
1,SES-021125,STN-012,VH-005,10-19-2025 17:10:08,10-19-2025 18:40:08,13.69,DC Fast Charge,1.92,debit_card,CHAdeMO,U-4434
2,SES-026798,STN-007,VEH#0005,12/30/2025 07:30,12/30/2025 08:21,9.62,Level 1,3.56,Credit Card,CCS,U-6845
3,SES-008299,STN-006,VH-005,04-28-2025 02:35:41,04-28-2025 04:52:41,70.99,DC Fast Charge,26.27,app_wallet,Tesla Supercharger,U-7854
4,SES-018503,STN-012,VH-011,2025-09-14 14:28:19,2025-09-14 16:39:19,10.15,Level 2,2.84,credit card,CCS,U-2903


In [ ]:
stations = pd.read_csv("data/station_locations.csv")
print("station_locations.csv")
print(f"  Shape: {stations.shape}")
stations

station_locations.csv
  Shape: (21, 8)


,station_id,station_name,city,state,zip_code,latitude,longitude,region
0,STN-001,Downtown Charging Hub,Charlottesville,VA,22902,38.0293,-78.4767,Central
1,STN-002,University Ave Station,Charlottesville,VA,22903,38.0336,-78.5080,Central
2,STN-003,Pantops Charging Plaza,Charlottesville,VA,22911,38.0280,-78.4490,East
3,STN-004,Barracks Road Station,Charlottesville,VA,22903,38.0440,-78.5070,West
4,STN-005,Rivanna Station,Charlottesville,Virginia,22911,38.0520,-78.4430,East
5,STN-006,5th Street Hub,Charlottesville,VA,22902,38.0200,-78.4880,Central
6,STN-007,Rio Road Charging,Charlottesville,VA,22901,38.0700,-78.4800,North
7,STN-008,Hydraulic Rd Station,Charlottesville,VA,22901,38.0560,-78.4990,North
8,STN-009,Seminole Trail Plaza,Charlottesville,VA,22901,38.0800,-78.4850,North
9,STN-010,Avon Street Hub,Charlottesville,virginia,22902,38.0100,-78.4750,South


In [ ]:
vehicles = pd.read_csv("data/vehicle_types.csv")
print("vehicle_types.csv")
print(f"  Shape: {vehicles.shape}")
vehicles.head(10)

vehicle_types.csv
  Shape: (42, 6)


,vehicle_id,vehicle_name,vehicle_class,connector_type,battery_kwh,manufacturer
0,VH-001,Tesla Model 3,Sedan,CCS,75.0,Tesla
1,VEH#0001,TESLA MODEL 3,SEDAN,CCS,NaN,TESLA
2,V_tesla_model_3,tesla model 3,sedan,ccs,75.0,tesla
3,VH-002,Tesla Model Y,SUV,CCS,NaN,Tesla
4,VEH#0002,TESLA MODEL Y,SUV,CCS,NaN,TESLA
5,V_tesla_model_y,tesla model y,suv,ccs,75.0,tesla
6,VH-003,Tesla Model S,Sedan,CCS,100.0,Tesla
7,VEH#0003,TESLA MODEL S,suv,CCS,NaN,TESLA
8,V_tesla_model_s,tesla model s,sedan,ccs,100.0,tesla
9,VH-004,Chevrolet Bolt EV,Hatchback,CCS,65.0,Chevrolet


In [ ]:
operators = pd.read_csv("data/grid_operators.csv")
print("grid_operators.csv")
operators

grid_operators.csv


,operator_id,operator_name,city,state,avg_daily_capacity_mw,peak_capacity_mw,cost_per_kwh
0,GO-CVL,Charlottesville Energy Cooperative,Charlottesville,VA,120,240,0.12
1,GO-DOM,Dominion Energy Virginia,Richmond,VA,450,800,0.11
2,GO-APC,Appalachian Power,Roanoke,VA,300,~550,0.13
3,GO-REC,Rappahannock Electric,Fredericksburg,VA,200,400,0.14
4,GO-NRG,Northern Virginia Electric,Fairfax,VA,380,700,NaN


In [ ]:
conn = sqlite3.connect("data/energy_and_demand.db")
demand = pd.read_sql("SELECT * FROM daily_demand_summary LIMIT 5", conn)
grid = pd.read_sql("SELECT * FROM grid_capacity_levels LIMIT 5", conn)
conn.close()

print("daily_demand_summary:")
print(demand)
print("\ngrid_capacity_levels:")
print(grid)

daily_demand_summary:
         date station_id  total_sessions  total_kwh_delivered  \
0  2025-01-01    STN-001               5               126.31   
1  2025-01-01    STN-002               6               212.06   
2  2025-01-01    STN-003               3                77.90   
3  2025-01-01    STN-004               7               274.85   
4  2025-01-01    STN-005               8               154.57   

   avg_session_duration_min  peak_hour  revenue_usd  
0                      61.5          9        32.08  
1                      69.0         13        29.18  
2                      85.8         14        18.57  
3                      65.6         22        94.37  
4                      31.5          9        36.30  

grid_capacity_levels:
         date operator_id  available_capacity_mw  load_pct  outage_flag  \
0  2025-01-01      GO-CVL                  239.2      50.7            0   
1  2025-01-01      GO-DOM                  186.6      51.1            1   
2  2025-01-01  

## Data Cleaning Pipeline


## Part 1 — Cleaning the session data

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import os

sessions = pd.read_csv("data/charging_sessions.csv")
print(f"Raw sessions: {sessions.shape}")
sessions.head()


Raw sessions: (27451, 11)


,session_id,station_id,vehicle_id,session_start,session_end,kwh_delivered,session_type,cost_usd,payment_method,connector_used,user_id
0,SES-005321,STN-013,VH-008,2025-03-16 17:06:25,2025-03-16 18:30:25,1.67,Level 1,0.53,credit card,J1772,U-4469
1,SES-021125,STN-012,VH-005,10-19-2025 17:10:08,10-19-2025 18:40:08,13.69,DC Fast Charge,1.92,debit_card,CHAdeMO,U-4434
2,SES-026798,STN-007,VEH#0005,12/30/2025 07:30,12/30/2025 08:21,9.62,Level 1,3.56,Credit Card,CCS,U-6845
3,SES-008299,STN-006,VH-005,04-28-2025 02:35:41,04-28-2025 04:52:41,70.99,DC Fast Charge,26.27,app_wallet,Tesla Supercharger,U-7854
4,SES-018503,STN-012,VH-011,2025-09-14 14:28:19,2025-09-14 16:39:19,10.15,Level 2,2.84,credit card,CCS,U-2903


In [ ]:
# Step 1: Drop exact duplicates
before = len(sessions)
sessions = sessions.drop_duplicates()
print(f"Dropped {before - len(sessions)} exact duplicates. Remaining: {len(sessions)}")

Dropped 486 exact duplicates. Remaining: 26965


In [ ]:
# Step 2: Standardize station IDs
import re

def standardize_station_id(sid):
    match = re.search(r'\d+', str(sid))
    if match:
        return f"STN-{match.group().zfill(3)}"
    return sid

sessions["station_id"] = sessions["station_id"].apply(standardize_station_id)
print("Station IDs after fix:", sorted(sessions["station_id"].unique()))

Station IDs after fix: ['STN-001', 'STN-002', 'STN-003', 'STN-004', 'STN-005', 'STN-006', 'STN-007', 'STN-008', 'STN-009', 'STN-010', 'STN-011', 'STN-012', 'STN-013', 'STN-014', 'STN-015', 'STN-016', 'STN-017', 'STN-018']


In [ ]:
# Step 3: Parse dates — build a multi-format parser

# replace placeholders
sessions["session_start"] = sessions["session_start"].replace(["NULL", "N/A", "NaN", "None", "nan"], np.nan)
sessions["session_end"] = sessions["session_end"].replace(["NULL", "N/A", "NaN", "None", "nan"], np.nan)

# parse mixed date formats
sessions["session_start"] = pd.to_datetime(sessions["session_start"], errors="coerce", format="mixed")
sessions["session_end"] = pd.to_datetime(sessions["session_end"], errors="coerce", format="mixed")

print("session_start nulls:", sessions["session_start"].isna().sum())
print("session_end nulls:", sessions["session_end"].isna().sum())

session_start nulls: 0
session_end nulls: 0


The timestamp columns contained mixed date formats, so we used pd.to_datetime(..., format="mixed") to standardize them. This successfully parsed all valid values, leaving no missing timestamps.

In [ ]:
# Step 4: Fix cost column — strip $ and convert
sessions["cost_usd"] = sessions["cost_usd"].replace(["NULL", "N/A", "NaN", "None", "nan"], np.nan)
sessions["cost_usd"] = sessions["cost_usd"].replace(r"[$,]", "", regex=True)
sessions["cost_usd"] = pd.to_numeric(sessions["cost_usd"], errors="coerce")

print("Missing cost values:", sessions["cost_usd"].isna().sum())
sessions[["cost_usd"]].head()

Missing cost values: 67


,cost_usd
0,0.53
1,1.92
2,3.56
3,26.27
4,2.84


The cost_usd column contained currency symbols and inconsistent formatting. We removed $ and commas using regex and converted the column to numeric. A small number of missing values (67 rows) remained and were retained for now.

In [ ]:
# Step 5: Fix kWh — handle negatives

# replace placeholders
sessions["kwh_delivered"] = sessions["kwh_delivered"].replace(
    ["NULL", "N/A", "NaN", "None", "nan"], np.nan
)

# convert to numeric
sessions["kwh_delivered"] = pd.to_numeric(sessions["kwh_delivered"], errors="coerce")

# check how many negative values
print("Negative kWh values:", (sessions["kwh_delivered"] < 0).sum())

# fix negatives (take absolute value)
sessions["kwh_delivered"] = sessions["kwh_delivered"].abs()

# check missing values
print("Missing kWh values:", sessions["kwh_delivered"].isna().sum())

sessions[["kwh_delivered"]].head()


Negative kWh values: 535
Missing kWh values: 90


,kwh_delivered
0,1.67
1,13.69
2,9.62
3,70.99
4,10.15


The kwh_delivered column was converted to numeric and checked for data entry errors. Negative values (543 rows) were identified and corrected by taking the absolute value, as energy delivered cannot be negative. A small number of missing values (90 rows) remained and were retained for later handling.

In [ ]:
# Step 6: Handle missing values
# Replace string nulls with actual NaN

null_strings = ["NULL", "N/A", "NaN", "nan", "None", ""]

sessions = sessions.replace(null_strings, np.nan)

# check missing values across all columns
print("Missing values by column:")
print(sessions.isna().sum())

Missing values by column:
session_id         0
station_id         0
vehicle_id        77
session_start      0
session_end        0
kwh_delivered     90
session_type       0
cost_usd          67
payment_method     0
connector_used    66
user_id            0
dtype: int64


String representations of missing values (e.g., "NULL", "N/A") were replaced with actual NaN values across the dataset to ensure consistency. After cleaning, only a small number of missing values remained in a few columns, which were retained for later handling during analysis.

## Part 2 — Vehicle ID consolidation

In [ ]:
vehicles = pd.read_csv("data/vehicle_types.csv")
print(f"Vehicle reference: {vehicles.shape}")
vehicles

Vehicle reference: (42, 6)


,vehicle_id,vehicle_name,vehicle_class,connector_type,battery_kwh,manufacturer
0,VH-001,Tesla Model 3,Sedan,CCS,75.0,Tesla
1,VEH#0001,TESLA MODEL 3,SEDAN,CCS,NaN,TESLA
2,V_tesla_model_3,tesla model 3,sedan,ccs,75.0,tesla
3,VH-002,Tesla Model Y,SUV,CCS,NaN,Tesla
4,VEH#0002,TESLA MODEL Y,SUV,CCS,NaN,TESLA
5,V_tesla_model_y,tesla model y,suv,ccs,75.0,tesla
6,VH-003,Tesla Model S,Sedan,CCS,100.0,Tesla
7,VEH#0003,TESLA MODEL S,suv,CCS,NaN,TESLA
8,V_tesla_model_s,tesla model s,sedan,ccs,100.0,tesla
9,VH-004,Chevrolet Bolt EV,Hatchback,CCS,65.0,Chevrolet


In [ ]:
# Build your vehicle ID mapping
# vehicle_map = { "VH-001": "Tesla Model 3", "VEH#0001": "Tesla Model 3", ... }

#Step 1: standardize vehicle_id format
sessions["vehicle_id"] = sessions["vehicle_id"].str.strip()
sessions["vehicle_id"] = sessions["vehicle_id"].str.upper()
vehicles["vehicle_id"] = vehicles["vehicle_id"].str.strip()
vehicles["vehicle_id"] = vehicles["vehicle_id"].str.upper()

# Step 2: Build mapping from vehicle_id -> clean vehicle_name

# clean the vehicle_name column (standardize formatting)
vehicles["vehicle_name_clean"] = vehicles["vehicle_name"].str.lower().str.strip()

# create mapping dictionary
vehicle_map = dict(zip(vehicles["vehicle_id"], vehicles["vehicle_name_clean"]))

# preview mapping
list(vehicle_map.items())[:10]

[('VH-001', 'tesla model 3'),
 ('VEH#0001', 'tesla model 3'),
 ('V_TESLA_MODEL_3', 'tesla model 3'),
 ('VH-002', 'tesla model y'),
 ('VEH#0002', 'tesla model y'),
 ('V_TESLA_MODEL_Y', 'tesla model y'),
 ('VH-003', 'tesla model s'),
 ('VEH#0003', 'tesla model s'),
 ('V_TESLA_MODEL_S', 'tesla model s'),
 ('VH-004', 'chevrolet bolt ev')]

In [ ]:
#Step 3: check results
# standardize vehicle_id first
sessions["vehicle_id"] = sessions["vehicle_id"].str.strip().str.upper()

# apply mapping
sessions["vehicle_name"] = sessions["vehicle_id"].map(vehicle_map)
sessions[["vehicle_id", "vehicle_name"]].head(10)

,vehicle_id,vehicle_name
0,VH-008,hyundai ioniq 5
1,VH-005,chevrolet bolt euv
2,VEH#0005,chevrolet bolt euv
3,VH-005,chevrolet bolt euv
4,VH-011,bmw ix
5,VH-004,chevrolet bolt ev
6,VH-008,hyundai ioniq 5
7,VH-001,tesla model 3
8,V_FORD_F-150_LIGHTNING,ford f-150 lightning
9,VH-003,tesla model s


In [ ]:
#Step 4: Check for unmapped values
print("Unmapped vehicle IDs:", sessions["vehicle_name"].isna().sum())

Unmapped vehicle IDs: 77


In [ ]:
print("Missing vehicle_id:", sessions["vehicle_id"].isna().sum())
print("Missing vehicle_name:", sessions["vehicle_name"].isna().sum())

Missing vehicle_id: 77
Missing vehicle_name: 77


After applying the vehicle ID mapping, 77 session records remained unmapped. These corresponded to rows with missing vehicle_id values rather than unmatched formatting, so they could not be assigned a canonical vehicle name.

In [ ]:
# Apply mapping to sessions
# sessions["vehicle_name"] = sessions["vehicle_id"].map(vehicle_map)

# standardize IDs first
sessions["vehicle_id"] = sessions["vehicle_id"].str.strip().str.upper()

# apply mapping
sessions["vehicle_name"] = sessions["vehicle_id"].map(vehicle_map)

# check results
sessions[["vehicle_id", "vehicle_name"]].head(10)

# check how many are still unmapped
print("Unmapped vehicle IDs:", sessions["vehicle_name"].isna().sum())


The vehicle ID mapping was applied to the session data to create a standardized vehicle_name column. This allows different ID variants representing the same vehicle to be consolidated for accurate analysis.

## Part 3 — Load into SQLite

In [ ]:
# conn = sqlite3.connect("ev_analytics.db")
# sessions_clean.to_sql("sessions", conn, if_exists="replace", index=False)
# stations_clean.to_sql("stations", conn, if_exists="replace", index=False)
# conn.close()

# load tables
stations = pd.read_csv("data/station_locations.csv")
vehicles = pd.read_csv("data/vehicle_types.csv")

# connect to SQLite database
conn = sqlite3.connect("ev_analytics.db")

# save tables
sessions.to_sql("sessions", conn, if_exists="replace", index=False)
stations.to_sql("stations", conn, if_exists="replace", index=False)
vehicles.to_sql("vehicles", conn, if_exists="replace", index=False)

# close connection
conn.close()

print("Database created and tables loaded successfully!")

Database created and tables loaded successfully!


In [ ]:
# Save cleaned CSV for upload
sessions.to_csv("cleaned_sessions.csv", index=False)
print("CSV saved.")

CSV saved.


## Part 4 — Upload to GCS

In [ ]:
from google.colab import auth
auth.authenticate_user()
from google.cloud import storage

client = storage.Client(project="ds2002-492012")
bucket = client.bucket("ds2002-capstone-sp26-v2")

TEAM = "team-14"

sessions.to_csv("cleaned_sessions.csv", index=False)

for fname in ["cleaned_sessions.csv", "ev_analytics.db"]:
    blob = bucket.blob(f"{TEAM}/{fname}")
    blob.upload_from_filename(fname)
    print(f"Uploaded {fname}")

Uploaded cleaned_sessions.csv
Uploaded ev_analytics.db


## Part 5 — External API setup

In [ ]:
import requests

# Charlottesville coordinates: 38.03, -78.48
# Example: get daily temperature for January 2025

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 38.03,
    "longitude": -78.48,
    "start_date": "2025-01-01",
    "end_date": "2025-01-31",
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "timezone": "America/New_York"
}

resp = requests.get(url, params=params)
data = resp.json()

weather = pd.DataFrame({
    "date": data["daily"]["time"],
    "temp_max_f": [c * 9/5 + 32 for c in data["daily"]["temperature_2m_max"]],
    "temp_min_f": [c * 9/5 + 32 for c in data["daily"]["temperature_2m_min"]],
    "precip_mm": data["daily"]["precipitation_sum"]
})

weather.to_csv("weather_charlottesville.csv", index=False)
conn = sqlite3.connect("ev_analytics.db")
weather.to_sql("weather", conn, if_exists="replace", index=False)
conn.close()

print(f"Weather data: {weather.shape}")
weather.head()

Weather data: (31, 4)


,date,temp_max_f,temp_min_f,precip_mm
0,2025-01-01,47.30,38.66,0.0
1,2025-01-02,42.08,31.46,0.0
2,2025-01-03,45.14,26.96,0.0
3,2025-01-04,32.18,24.62,0.0
4,2025-01-05,39.56,22.64,4.2
